# Smart Farming: Vision + IndicTrans2 on one Colab runtime

Two FastAPI services run side by side, each behind its own Cloudflare quick tunnel:

| Service | Port | Endpoint |
|---|---|---|
| Vision models (`main:app`) | 8001 | your existing routes |
| IndicTrans2 translation | 8002 | `POST /translate`, `GET /health` |

Runtime: **GPU (T4)**. Run the cells top to bottom. Both URLs change on every restart.

### Cell 1: Mount Drive and sync the vision service code

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/Smart-Farming
!git clone --depth 1 --filter=blob:none --sparse https://github.com/manthan2876/Smart-Farming.git /content/Smart-Farming
%cd /content/Smart-Farming
!git sparse-checkout set model_service

SRC = '/content/Smart-Farming/model_service'
DST = '/content/drive/MyDrive/SmartFarming'
!rm -rf "{DST}"
!cp -r "{SRC}" "{DST}"
print("Vision service code synced to Drive.")

Mounted at /content/drive
Cloning into '/content/Smart-Farming'...
remote: Enumerating objects: 413, done.
remote: Counting objects: 100% (413/413), done.
remote: Compressing objects: 100% (359/359), done.
remote: Total 413 (delta 3), reused 344 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (413/413), 3.04 MiB | 18.23 MiB/s, done.
Resolving deltas: 100% (3/3), done.
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 24.31 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Smart-Farming
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 59 (delta 1), reused 50 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 164.04 MiB | 25.75 MiB/s, done.
Resolving deltas: 100% (1/1),

### Cell 2: Install dependencies (vision + translation)

In [2]:
!pip install -q -r /content/drive/MyDrive/SmartFarming/requirements.txt

# transformers must stay on 4.x: IndicTransToolkit and the IndicTrans2 remote code break on 5.x
!pip install -q "transformers>=4.44,<5" sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit

# Separate folder for the translation server (the vision folder above gets wiped on every sync)
!mkdir -p /content/drive/MyDrive/IndicTransServer

# Sanity check: this must print "toolkit import OK" before you continue
!python -c "import transformers, huggingface_hub; print('transformers', transformers.__version__, '| hub', huggingface_hub.__version__); from IndicTransToolkit.processor import IndicProcessor; print('toolkit import OK')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
di

### Cell 3: Download cloudflared

In [3]:
#@title Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

### Cell 4: Write the IndicTrans2 server file to Drive

In [9]:
%%writefile /content/drive/MyDrive/IndicTransServer/translate_server.py
import os, json, threading
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

LANGS = {"hi": "hin_Deva", "gu": "guj_Gujr"}
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
CACHE_FILE = "/content/drive/MyDrive/indictrans_cache.json"

# Works with or without a GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_BEAMS = 4 if DEVICE == "cuda" else 2
CHUNK = 16 if DEVICE == "cuda" else 8
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 2)
print(f"[Init] Device: {DEVICE}")

print("[Init] Loading IndicTrans2 tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE).eval()
processor = IndicProcessor(inference=True)
lock = threading.Lock()

# Persistent translation cache on Drive
cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            cache = json.load(f)
        print(f"[Cache] Loaded {len(cache)} cached translations from Drive.")
    except Exception as e:
        print(f"[Cache] Could not load cache: {e}")

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    except Exception:
        pass

def run(texts, tgt):
    batch = processor.preprocess_batch(texts, src_lang="eng_Latn", tgt_lang=tgt)
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=256,
                       return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, num_beams=NUM_BEAMS, max_length=256, use_cache=False)
    return processor.postprocess_batch(
        tokenizer.batch_decode(out, skip_special_tokens=True), lang=tgt)

# Warmup so the first real request is faster
run(["Field health looks good."], "hin_Deva")
print("[Init] Model warmed up and ready.")

app = FastAPI(title="IndicTrans2 Translation Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

class TranslationRequest(BaseModel):
    texts: list[str]
    target: str  # "hi" or "gu"

@app.get("/health")
def health():
    return {"status": "healthy", "service": "indictrans2", "device": DEVICE}

@app.post("/translate")
def translate(req: TranslationRequest):
    lang = req.target.lower()
    tgt = LANGS.get(lang)
    if not tgt:
        return {"translations": req.texts, "error": f"Unsupported target {req.target}"}

    def key(t):
        return f"{lang}:{t}"

    with lock:
        todo = [t for t in dict.fromkeys(req.texts) if key(t) not in cache]
        for i in range(0, len(todo), CHUNK):
            chunk = todo[i:i + CHUNK]
            for s, d in zip(chunk, run(chunk, tgt)):
                cache[key(s)] = d
        if todo:
            save_cache()
        return {"translations": [cache.get(key(t), t) for t in req.texts]}

Overwriting /content/drive/MyDrive/IndicTransServer/translate_server.py


### Cell 5: Start both servers in the background

In [10]:
import os, subprocess, time, signal, requests
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN in Colab Secrets (key icon) and enable Notebook access"
print("HF_TOKEN loaded")

def kill_matching(pattern):
    pids = subprocess.run(["pgrep", "-f", pattern], capture_output=True, text=True).stdout.split()
    for pid in pids:
        try:
            os.kill(int(pid), signal.SIGTERM)
        except Exception:
            pass

for pat in ["uvicorn main:app", "uvicorn translate_server:app", "cloudflared tunnel"]:
    kill_matching(pat)
time.sleep(2)

os.makedirs("/content/hf_cache", exist_ok=True)

def start(cmd, cwd, log):
    env = {**os.environ, "HF_TOKEN": HF_TOKEN, "HF_HOME": "/content/hf_cache"}
    return subprocess.Popen(cmd, cwd=cwd, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

vision_proc = start(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8001"],
    "/content/drive/MyDrive/SmartFarming", "/content/vision.log")

translate_proc = start(
    ["uvicorn", "translate_server:app", "--host", "0.0.0.0", "--port", "8002"],
    "/content/drive/MyDrive/IndicTransServer", "/content/translate.log")

def tail(path, n=30):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

def wait_ready(name, url, proc, log, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            print(f"{name}: process exited (code {proc.returncode}). Last log lines:\n{tail(log)}")
            return False
        try:
            if requests.get(url, timeout=2).status_code < 500:
                print(f"{name}: ready after {int(time.time() - t0)}s")
                return True
        except Exception:
            pass
        time.sleep(3)
    print(f"{name}: not ready after {timeout}s. Last log lines:\n{tail(log)}")
    return False

vision_ok = wait_ready("vision", "http://127.0.0.1:8001/docs", vision_proc, "/content/vision.log")
translate_ok = wait_ready("translate", "http://127.0.0.1:8002/health", translate_proc, "/content/translate.log")

HF_TOKEN loaded
vision: ready after 9s
translate: ready after 12s


### Cell 6: One Cloudflare tunnel per port

In [11]:
import re, json, time, subprocess

def open_tunnel(port, timeout=60):
    log = f"/content/tunnel_{port}.log"
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=open(log, "w"), stderr=subprocess.STDOUT)
    end = time.time() + timeout
    while time.time() < end:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open(log).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError(f"No tunnel URL for port {port}. Check {log}")

vision_tunnel, VISION_URL = open_tunnel(8001)
translate_tunnel, TRANSLATE_URL = open_tunnel(8002)

print("==========================================")
print("VISION_SERVICE_URL    =", VISION_URL)
print("TRANSLATE_SERVICE_URL =", TRANSLATE_URL)
print("==========================================")

# Keep the latest pair on Drive for easy copy-paste later
json.dump({"vision": VISION_URL, "translate": TRANSLATE_URL},
          open("/content/drive/MyDrive/service_urls.json", "w"))

VISION_SERVICE_URL    = https://scanned-norm-limousines-parade.trycloudflare.com
TRANSLATE_SERVICE_URL = https://hazards-bible-sequences-rider.trycloudflare.com


### Cell 7: Test both public URLs

In [13]:
import requests, time

def retry(fn, tries=20, delay=3):
    for _ in range(tries):
        try:
            r = fn()
            if r.status_code < 500:
                return r
        except Exception:
            pass
        time.sleep(delay)   # new tunnels can take a few seconds to start resolving
    return None

r = retry(lambda: requests.get(f"{TRANSLATE_URL}/health", timeout=10))
print("translate health:", r.json() if r is not None else "FAILED")

for lang in ["hi", "gu"]:
    r = retry(lambda: requests.post(
        f"{TRANSLATE_URL}/translate",
        json={"texts": ["Your crop shows signs of leaf blight."], "target": lang},
        timeout=60))
    print(lang, "->", r.json() if r is not None else "FAILED")

r = retry(lambda: requests.get(f"{VISION_URL}/docs", timeout=10))
print("vision status:", r.status_code if r is not None else "FAILED")

translate health: {'status': 'healthy', 'service': 'indictrans2', 'device': 'cpu'}
hi -> {'translations': ['आपकी फसल में पत्ती के रोग के संकेत दिखाई देते हैं।']}
gu -> {'translations': ['તમારા પાકમાં પાંદડાનો સોજો આવવાના સંકેતો દેખાય છે.']}
vision status: 200


### Cell 8 (optional): Logs and status, run this if something fails

In [15]:
for name in ["vision.log", "translate.log", "tunnel_8001.log", "tunnel_8002.log"]:
    print(f"\n===== {name} (last 25 lines) =====")
    !tail -25 /content/{name}
print("\nGPU memory:")
!nvidia-smi || echo "No GPU on this runtime (running on CPU)"


===== vision.log (last 25 lines) =====
INFO:     Started server process [3230]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)
[ModelService] Pre-loading vision models into RAM in background...
[ModelService] Vision models successfully loaded and ready in RAM.
INFO:     127.0.0.1:56246 - "GET /docs HTTP/1.1" 200 OK
INFO:     35.237.84.211:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     35.237.84.211:0 - "GET /docs HTTP/1.1" 200 OK

===== translate.log (last 25 lines) =====
I0000 00:00:1791459210.549603    3231 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791459210.619730    3231 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I